# AdaFace Noise Study - Colab Launcher

This notebook is the **EXECUTION LAUNCHER** for the AdaFace Noise Study.
It orchestrates Google Colab GPU training, pulls the exact source code from GitHub, syncs heavy datasets from Google Drive, and pushes results back.

## 1. System Verification & Mount

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Obtain Source Code (Git)

In [ ]:
import os
import sys
SOURCE_GIT_REF = "colab-0pct-v1"
REPO_URL = "https://github.com/Raja2027/adaFace-noise-study.git"

if not os.path.exists('/content/adaFace-noise-study'):
    !git clone {REPO_URL} /content/adaFace-noise-study

os.chdir('/content/adaFace-noise-study')
!git fetch origin

# Capture LAUNCHER commit before checking out source
launcher_commit_raw = !git rev-parse HEAD
LAUNCHER_COMMIT = launcher_commit_raw[0].strip()
print(f"Launcher commit: {LAUNCHER_COMMIT}")

# Checkout the strictly pinned source tag
!git checkout {SOURCE_GIT_REF}
!git submodule update --init --recursive

# Verify exact hashes
source_commit = !git rev-parse HEAD
source_commit = source_commit[0].strip()
if source_commit != "036d6eaf98ad8d1132662cdccef945cfbcdb3ea8":
    print(f"CRITICAL: Source commit mismatch! Expected 036d6eaf..., got {source_commit}")
    sys.exit(1)
    
adaface_commit = !git -C third_party/AdaFace rev-parse HEAD
adaface_commit = adaface_commit[0].strip()
if adaface_commit != "c60eaa786a42c03444f3df7096dbaf9d57ae010d":
    print(f"CRITICAL: AdaFace commit mismatch! Expected c60eaa7..., got {adaface_commit}")
    sys.exit(1)
    
print("Hash verification passed.")

## 3. Data Sync & Environment Setup

In [ ]:
!python colab/setup_colab.py

## 4. Benchmark Physical Batch Size (Optional)

In [ ]:
!python colab/benchmark_batch_size.py

## 5. FINAL PREFLIGHT VERIFICATION
**DO NOT START TRAINING UNLESS THIS SUCCEEDS.**

In [ ]:
!python colab/preflight_colab.py

## 6. Execute Training (0% Baseline)
Starts the FULL 0% NOISE BASELINE experiment natively syncing to Drive.

In [ ]:
# Patch the run manifest dynamically to record BOTH launcher and source commits
import re
with open('colab/train_colab.py', 'r') as f:
    content = f.read()
content = content.replace(
    "'git_commit': git_hash,",
    f"'LAUNCHER_GIT_COMMIT': '{LAUNCHER_COMMIT}',\n        'SOURCE_GIT_COMMIT': git_hash,"
)
with open('colab/train_colab.py', 'w') as f:
    f.write(content)

!python colab/train_colab.py \
    --config configs/noise_0.yaml \
    --sync_dir /content/drive/MyDrive/adaFace-noise-study/checkpoints/noise_0/seed_42